#### 13.Image Models: EfficientNet vs Simple CNN

### Architecture Justification

**EfficientNetB0 — Transfer Learning:**
- Pre-trained on ImageNet (1.28M images, 1000 classes)
- Frozen base = preserves learned features (edges, textures, shapes)
- Added GlobalAveragePooling2D → Dense(64) → Dropout(0.3) → sigmoid
- Dropout = 0.3 chosen to regularize without underfitting

**Simple CNN — Baseline:**
- 2 Conv layers (32→64 filters) trained from scratch
- Used only to prove Transfer Learning superiority
- Expected to underperform due to small dataset size

**Conclusion:**
EfficientNet leverages pretrained knowledge → higher accuracy with fewer epochs
Simple CNN trains from zero → weaker on small datasets




In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D

# --- Pretrained EfficientNetB0 ---
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze pretrained weights

model_image = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_image.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_image = model_image.fit(
    X_img_train,
    y_train,
    epochs=6,
    batch_size=16,
    validation_data=(X_img_test, y_test)
)

image_pred = (model_image.predict(X_img_test) > 0.5).astype(int).ravel()
image_acc = accuracy_score(y_test, image_pred)
print("EfficientNet (Pretrained) Image Model Accuracy:", image_acc)


# --- Fine-tuning: Unfreeze last 20 layers ---
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model_image.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # learning rate صغير جداً
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nFine-tuning EfficientNet (last 20 layers)...")
history_finetune = model_image.fit(
    X_img_train,
    y_train,
    epochs=3,
    batch_size=16,
    validation_data=(X_img_test, y_test)
)

image_pred = (model_image.predict(X_img_test) > 0.5).astype(int).ravel()
image_acc = accuracy_score(y_test, image_pred)
print("EfficientNet (After Fine-tuning) Accuracy:", image_acc)


# --- Compare with Simple CNN ---
model_cnn_simple = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_cnn_simple.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_cnn_simple.fit(
    X_img_train,
    y_train,
    epochs=6,
    batch_size=16,
    validation_data=(X_img_test, y_test)
)

cnn_simple_pred = (model_cnn_simple.predict(X_img_test) > 0.5).astype(int).ravel()
cnn_simple_acc = accuracy_score(y_test, cnn_simple_pred)

print("\n=== Image Model Comparison ===")
print(f"Simple CNN Accuracy:               {cnn_simple_acc:.4f}")
print(f"EfficientNet (After Fine-tuning):  {image_acc:.4f}")
print("EfficientNet is better: pretrained on ImageNet + fine-tuned on our data")